In [1]:
# Enable autoreload when modifying files
%load_ext autoreload
%autoreload 2

In [2]:
# Convert existing csvs to parquet
from icare_risk.utils import convert_csvs_to_parquet
path = '/app/data/mock/sirs_test'
convert_csvs_to_parquet(directory_path=path, delete_originals=False)

Converting: episodes.csv -> episodes.parquet
Converting: pathology.csv -> pathology.parquet
Converting: problems.csv -> problems.parquet
Converting: vitals.csv -> vitals.parquet
Successfully converted 4 files.


In [12]:
# -------------------------------------------------------
# Load mock data
# -------------------------------------------------------
from icare_risk.core.dataset import ClinicalDataset
from icare_risk.config.settings import (
    EPISODE_CONFIG, TABLE_CONFIG,
    get_dataset_paths
)

# Toggle to select dataset
paths_mock = get_dataset_paths(env='mock', dataset_name='sirs_test')

# Initializes DuckDB and automatically maps the CSVs to the standard views
dataset = ClinicalDataset(
    **paths_mock,
    episode_config=EPISODE_CONFIG,
    table_config=TABLE_CONFIG
)

# Now run your standard pipeline
vitals_df = dataset.get_current_stay('vitals')
pathology_df = dataset.get_current_stay('pathology')

print(vitals_df.shape)
print(pathology_df.shape)

{'table_paths': {'vitals': '/app/data/mock/sirs_test/vitals.csv', 'pathology': '/app/data/mock/sirs_test/pathology.csv', 'problems': '/app/data/mock/sirs_test/problems.csv'}, 'episodes_path': '/app/data/mock/sirs_test/episodes.csv'}
(15, 7)
(7, 7)


In [9]:
# -------------------------------------------------------
# Manually call each function
# -------------------------------------------------------
# Libraries
from icare_risk.core.inspect import  print_df
from icare_risk.phenotypes.sirs import derive_sirs_tachycardia
from icare_risk.phenotypes.sirs import derive_sirs_tachypnea
from icare_risk.phenotypes.sirs import derive_sirs_abnormal_temp
from icare_risk.phenotypes.sirs import derive_sirs_abnormal_wbc

# Features
df1 = derive_sirs_tachycardia(dataset)
df2 = derive_sirs_tachypnea(dataset)
df3 = derive_sirs_abnormal_temp(dataset)
df4 = derive_sirs_abnormal_wbc(dataset)

# Display
print_df(df1)
print_df(df2)
print_df(df3)
print_df(df4)


=== Shape: (5, 3) | Counts in 'sirs_tachycardia_flag': {0: 3, 1: 2} ===
   SUBJECT  std_admission_time  sirs_tachycardia_flag
0      101 2026-09-01 08:00:00                      1
1      102 2026-09-02 08:00:00                      0
2      103 2026-09-03 08:00:00                      0
3      104 2026-09-04 08:00:00                      0
4      105 2026-09-05 08:00:00                      1

=== Shape: (5, 3) | Counts in 'sirs_tachypnea_flag': {0: 3, 1: 2} ===
   SUBJECT  std_admission_time  sirs_tachypnea_flag
0      101 2026-09-01 08:00:00                    1
1      102 2026-09-02 08:00:00                    0
2      103 2026-09-03 08:00:00                    0
3      104 2026-09-04 08:00:00                    1
4      105 2026-09-05 08:00:00                    0

=== Shape: (5, 3) | Counts in 'sirs_abnormal_temp_flag': {1: 3, 0: 2} ===
   SUBJECT  std_admission_time  sirs_abnormal_temp_flag
0      101 2026-09-01 08:00:00                        1
1      102 2026-09-02 08:00:00   

In [10]:
# --------------------------------------------------------------
# Automation from YAML
# --------------------------------------------------------------
# Libraries
from icare_risk.core.builder import build_feature_matrix
from icare_risk.core.utils import load_phenotypes_config

# Define .yaml path
yaml_path = '/app/src/icare_risk/config/phenotypes_syn.yaml'

# Load phenotypes for sirs
sirs_phenotypes_config = load_phenotypes_config(
    yaml_path=yaml_path, prefix='sirs'
)
print(sirs_phenotypes_config.keys())

# Compute feature matrix
final_features = build_feature_matrix(dataset, sirs_phenotypes_config)

print("\n--- SUM ---")
print(final_features.sum(axis=0))
print("\n--- FINAL FEATURE MATRIX ---")
print(final_features)

dict_keys(['sirs_tachycardia_flag', 'sirs_tachypnea_flag', 'sirs_abnormal_temp_flag', 'sirs_abnormal_wbc_flag'])
Computing: sirs_tachycardia_flag...
Computing: sirs_tachypnea_flag...
Computing: sirs_abnormal_temp_flag...
Computing: sirs_abnormal_wbc_flag...

--- SUM ---
sirs_tachycardia_flag      2
sirs_tachypnea_flag        2
sirs_abnormal_temp_flag    3
sirs_abnormal_wbc_flag     3
dtype: int64

--- FINAL FEATURE MATRIX ---
                             sirs_tachycardia_flag  sirs_tachypnea_flag  \
SUBJECT std_admission_time                                                
101     2026-09-01 08:00:00                      1                    1   
102     2026-09-02 08:00:00                      0                    0   
103     2026-09-03 08:00:00                      0                    0   
104     2026-09-04 08:00:00                      0                    1   
105     2026-09-05 08:00:00                      1                    0   

                             sirs_abnormal_t